In [ ]:
# Portable project paths. Set TLS_PROJECT_ROOT to the directory containing the input data.
import os
from pathlib import Path
PROJECT_ROOT = Path(os.environ.get("TLS_PROJECT_ROOT", ".")).resolve()


NMF前10个top

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import networkx as nx
from collections import defaultdict
import re
import pandas as pd
import numpy as np
import os
import matplotlib.lines as mlines
import time
import scanpy as sc
from matplotlib import pyplot as plt
import colorsys  # HLS 调整颜色
import math

# ========================== 全局可调参数（尺寸/粗细/布局） ==========================
TOP_K = 20
NODE_LABEL_FONT_SIZE = 20  # 圆圈上的细胞类型文字大小
LEGEND_FONT_SIZE = 16      # 图注文字大小
TITLE_FONT_SIZE = 21       # 标题文字大小
EDGE_ALPHA = 0.95

# —— Adobe Illustrator 友好导出：PDF/SVG 保留可编辑文字，避免 Type 3 字体
mpl.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "axes.unicode_minus": False,
})

# —— 节点外圈描边
NODE_BORDER_WIDTH = 0.3

# —— 受配体对连线宽度（固定；所有细胞模块连线一样粗）
EDGE_WIDTH = 0.9

# —— 箭头
ARROWSIZE = 12
ARROWSTYLE = "-|>"

# —— 边上组别符号（△、★、+…）
DOT_SIZE = 70
DOT_EDGEWIDTH = 0.10
HALO_SCALE = 1.45
HALO_EDGEWIDTH = 0.90

# —— 仅当边太短/点太多时改用法线堆叠的阈值（像素）
DOT_SEP_PX = 16
END_MARGIN_PX = 18
STACK_SEP_PX = 14
MIN_INLINE_PX = 130

# —— 线型 marker 集合（不填充）
LINE_MARKERS = {"+", "x", "|", "_", "1", "2", "3", "4"}

# —— 图注（Legend）样式专用参数（更细描边）
LEGEND_MARKERSIZE = 8.5
LEGEND_GROUP_EDGEWIDTH_LINE = 0.45   # 线型标记
LEGEND_GROUP_EDGEWIDTH_FILL = 0.40   # 填充型标记
LEGEND_LR_LINEWIDTH = 2.2            # 受配体对彩色短横线

# —— 是否增强组别颜色对比度（HLS）
ENHANCE_GROUP_CONTRAST = True
HLS_LIGHTEN = 1.10
HLS_DARKEN  = 0.85
HLS_SAT_SCALE = 1.08

# —— 节点“最小间距 + 轻量排斥式松弛”
NODE_MIN_DIST = 0.35
REPEL_ITERS = 140
REPEL_STEP = 0.08
EXTRA_REPEL_PAIRS = {("JUN_CD16NK", "XCL1_CD16NK")}
EXTRA_REPEL_STRENGTH = 3.5

# —— 指定节点对的“最小角度分开”硬约束（圆形布局之后执行）
ANGULAR_SPREAD_PAIRS = {("JUN_CD16NK", "XCL1_CD16NK")}
PAIR_MIN_ANGLE_DEG = 30.0   # 至少 30° 的夹角
PAIR_RADIUS_MODE = "max"    # "keep" 保持各自半径；"max" 用两者半径的最大值

# ========================== 组别与形状设置 ==========================
group_markers = {
    "N": "o", "AD": "s", "Inf": "^", "NAT": "D", "Pre": "v",
    "T": "X", "Met": "*", # "BT": "+"  #"TB": "H",
}

# ========================== 数据路径 ==========================
csv_directory = str(PROJECT_ROOT / "泛癌S/NMF/cellchat_sample/细胞特异相关性/疾病分组细胞特异相关性/重新算特异相关性/P值小于0.05相关性大于0.3/2025-10-23")

file_path = str(PROJECT_ROOT / "泛癌S/NMF/cellchat_sample/重新提取cellchat结果/cell_ligand_pairs_13_with_weights_2025_10_30.txt")
adata_path = str(PROJECT_ROOT / "泛癌S/h5ad/combined_adata_inner.h5ad")
save_dir = str(PROJECT_ROOT / "泛癌S/NMF/cellchat_sample/重新提取cellchat结果/模块网络图")

groups = [
    "Normal", "Autoimmune diseases", "Inflammation",
    "Normal adjacent tissue", "Precancerous condition",
    "Tumor", "Tumor_metastasis", # "Blood_Tumor",  #"Tumor_blood",
]
group_abbreviations = {
    "Normal": "N", "Autoimmune diseases": "AD", "Inflammation": "Inf",
    "Normal adjacent tissue": "NAT", "Precancerous condition": "Pre",
    "Tumor": "T", "Tumor_metastasis": "Met",# "Tumor_blood": "TB",
   # "Blood_Tumor": "BT",
}
group_color_full = {
    "Autoimmune diseases": "#F1BB72", "Inflammation": "#D6E7A3",
    "Normal": "#E5D2DD", "Normal adjacent tissue": "#57C3F3",
    "Precancerous condition": "#E59CC4", "Tumor": "#53A85F",
    "Tumor_metastasis": "#F3B1A0",# "Tumor_blood": "#476D87",
    #"Blood_Tumor": "#E95C59",
}
group_colors = {group_abbreviations[n]: group_color_full[n]
                for n in group_abbreviations if n in group_color_full}

def hex_to_rgb01(hx):
    hx = hx.lstrip("#")
    return tuple(int(hx[i:i+2], 16)/255.0 for i in (0, 2, 4))

def rgb01_to_hex(rgb):
    return "#{:02X}{:02X}{:02X}".format(
        int(max(0,min(1,rgb[0]))*255+0.5),
        int(max(0,min(1,rgb[1]))*255+0.5),
        int(max(0,min(1,rgb[2]))*255+0.5),
    )

def enhance_palette_by_hls(colors_dict, lighten=1.1, darken=0.85, sat_scale=1.08):
    keys = list(colors_dict.keys())
    new_map = {}
    for i, k in enumerate(keys):
        r, g, b = hex_to_rgb01(colors_dict[k])
        h, l, s = colorsys.rgb_to_hls(r, g, b)
        s = max(0, min(1, s * sat_scale))
        if i % 2 == 0: l = max(0, min(1, l * lighten))
        else:         l = max(0, min(1, l * darken))
        nr, ng, nb = colorsys.hls_to_rgb(h, l, s)
        new_map[k] = rgb01_to_hex((nr, ng, nb))
    return new_map

if ENHANCE_GROUP_CONTRAST:
    group_colors = enhance_palette_by_hls(group_colors, HLS_LIGHTEN, HLS_DARKEN, HLS_SAT_SCALE)

def build_group_legend_handles():
    group_legend = []
    for full, abbr in group_abbreviations.items():
        if abbr not in group_colors:
            continue
        mk = group_markers.get(abbr, "o")
        if mk in LINE_MARKERS:
            group_legend.append(
                mlines.Line2D([], [], marker=mk, linestyle='None',
                              markersize=LEGEND_MARKERSIZE, markerfacecolor='none',
                              markeredgecolor=group_colors[abbr],
                              markeredgewidth=LEGEND_GROUP_EDGEWIDTH_LINE,
                              label=f"{abbr}: {full}")
            )
        else:
            group_legend.append(
                mlines.Line2D([], [], marker=mk, linestyle='None',
                              markersize=LEGEND_MARKERSIZE,
                              markerfacecolor=group_colors[abbr],
                              markeredgecolor='black',
                              markeredgewidth=LEGEND_GROUP_EDGEWIDTH_FILL,
                              label=f"{abbr}: {full}")
            )
    return group_legend


def plot_group_symbol_legend(save_dir):
    handles = build_group_legend_handles()
    if not handles:
        return
    fig, ax = plt.subplots(figsize=(6.4, 3.0))
    ax.axis('off')
    ax.legend(handles=handles, title='Disease group', loc='center',
              fontsize=LEGEND_FONT_SIZE, title_fontsize=LEGEND_FONT_SIZE,
              frameon=False, ncol=1, handletextpad=0.6, labelspacing=0.45)
    os.makedirs(save_dir, exist_ok=True)
    fig.savefig(os.path.join(save_dir, "disease_group_symbol_legend.pdf"),
                bbox_inches='tight', dpi=600)
    plt.close(fig)

# ========================== 载入相关性数据（各疾病组） ==========================
group_data = {}
all_cell_types = set()
for group in groups:
    p = os.path.join(csv_directory, f"specificity_correlation_{group.replace(' ', '_')}.csv")
    if not os.path.exists(p):
        print(f"[warn] 未找到该组CSV，已跳过：{p}")
        continue
    df = pd.read_csv(p)
    group_data[group] = df[['Cell_Type_1', 'Cell_Type_2', 'Correlation', 'Adjusted_p_value']]
    all_cell_types.update(df['Cell_Type_1']); all_cell_types.update(df['Cell_Type_2'])

# ========================== 解析模块（含权重）与注释 ==========================
adata = sc.read_h5ad(adata_path)
cell_types = set(adata.obs['celltype_3'].dropna().unique())

celltype_mapping = {}
for ct3 in cell_types:
    subset = adata[adata.obs['celltype_3'] == ct3]
    if not subset.obs.empty:
        celltype_mapping[ct3] = subset.obs['celltype_2'].iloc[0]
celltype_2_categories = set(celltype_mapping.values())

# 调色板
shoupeitidui = [
    # Tableau/ColorBrewer-like qualitative palette: 区分度高，适合配体-受体连线。
    "#4E79A7", "#F28E2B", "#E15759", "#76B7B2", "#59A14F",
    "#B07AA1", "#FF9DA7", "#9C755F", "#1F77B4", "#FF7F0E",
    "#2CA02C", "#D62728", "#9467BD", "#8C564B", "#E377C2",
    "#7F7F7F", "#17BECF", "#BCBD22", "#393B79", "#637939",
]
xibaoleixing = [
 '#E5D2DD', '#53A85F', '#F1BB72', '#F3B1A0', '#D6E7A3', '#57C3F3', '#476D87',
  '#E95C59', '#E59CC4', '#AB3282', '#23452F', '#BD956A', '#8C549C', '#585658',
  '#9FA3A8', '#E0D4CA', '#5F3D69', '#C5DEBA', '#58A4C3', '#E4C755', '#F7F398',
  '#AA9A59', '#E63863', '#E39A35', '#C1E6F3', '#6778AE', '#91D0BE', '#B53E2B',
  '#712820', '#DCC1DD', '#CCE0F5',  '#CCC9E6', '#625D9E', '#68A180', '#3A6963',
  '#968175'
]
celltype_2_colors = {cat: xibaoleixing[i % len(xibaoleixing)] for i, cat in enumerate(celltype_2_categories)}

# 读取模块-相互作用
modules = defaultdict(list)
current_module = None
with open(file_path, 'r') as f:
    for line in f:
        line = line.strip()
        if line.startswith("Module:"):
            current_module = line.split(":")[1].strip()
        elif line.startswith("-") and current_module:
            m = re.match(r"-\s+(.+?)\s+\(weight:\s*([\d.]+)\)", line)
            if m:
                modules[current_module].append((m.group(1), float(m.group(2))))

module_cell_types = {}
module_ligand_receptors = {}
module_interactions = {}
module_cell_max_weights = {}

def extract_cell_type_from_string(s, cell_types):
    for ct in sorted(cell_types, key=len, reverse=True):
        if s.startswith(ct) and (len(s) == len(ct) or s[len(ct)] == '_'):
            return ct, s[len(ct):].lstrip('_')
    parts = s.split('_', 1)
    return (parts[0], parts[1] if len(parts) > 1 else "") if parts else ("", s)

for module, interactions in modules.items():
    cell_max_w = {}
    ct_set, lr_set, details = set(), set(), []
    for s, w in interactions:
        sender, rest = extract_cell_type_from_string(s, cell_types)
        if not sender: continue
        receiver, lr = extract_cell_type_from_string(rest, cell_types)
        if not receiver: continue

        cell_max_w[sender] = max(cell_max_w.get(sender, 0), w)
        cell_max_w[receiver] = max(cell_max_w.get(receiver, 0), w)

        gps = []
        for g in groups:
            df = group_data.get(g, None)
            if df is None: continue
            if (((df['Cell_Type_1']==sender)&(df['Cell_Type_2']==receiver)).any() or
                ((df['Cell_Type_1']==receiver)&(df['Cell_Type_2']==sender)).any()):
                gps.append(group_abbreviations[g])
        if gps:
            ct_set.update([sender, receiver]); lr_set.add(lr)
            details.append({"sender":sender, "receiver":receiver, "lr_pair":lr, "weight":w, "groups_present":gps})

    details = sorted(details, key=lambda x: x["weight"], reverse=True)[:TOP_K]
    module_cell_types[module] = ct_set
    module_ligand_receptors[module] = lr_set
    module_interactions[module] = details
    module_cell_max_weights[module] = cell_max_w

# ---------------------- 工具函数（像素/采样/法线堆叠） ----------------------
def px_to_data_vec(ax, p_disp, dx_px, dy_px):
    inv = ax.transData.inverted()
    x0, y0 = inv.transform(p_disp)
    x1, y1 = inv.transform((p_disp[0] + dx_px, p_disp[1] + dy_px))
    return np.array([x1 - x0, y1 - y0])

def polyline_length_px(poly_disp):
    d = np.diff(poly_disp, axis=0)
    if len(d) == 0: return 0.0
    return np.hypot(d[:,0], d[:,1]).sum()

def point_on_polyline_by_frac(verts_xy, frac):
    diffs = np.diff(verts_xy, axis=0)
    if len(diffs) == 0: return verts_xy[0]
    seg_len = np.hypot(diffs[:,0], diffs[:,1])
    total = seg_len.sum()
    if total == 0: return verts_xy[0]
    cum = np.concatenate([[0.0], np.cumsum(seg_len)])
    t = np.clip(frac * total, 0.0, total)
    k = int(np.searchsorted(cum, t, side="right") - 1)
    k = max(0, min(k, len(seg_len)-1))
    remain = t - cum[k]; u = 0.0 if seg_len[k] == 0 else remain / seg_len[k]
    return verts_xy[k] + u * diffs[k]

def point_on_poly_by_frac_disp(poly_disp, frac):
    d = np.diff(poly_disp, axis=0)
    seg = np.hypot(d[:,0], d[:,1]); total = seg.sum()
    if total == 0: return poly_disp[0]
    cum = np.concatenate([[0.0], np.cumsum(seg)])
    t = np.clip(frac * total, 0.0, total)
    k = int(np.searchsorted(cum, t, side="right") - 1)
    k = max(0, min(k, len(seg)-1))
    remain = t - cum[k]; u = 0.0 if seg[k] == 0 else remain / seg[k]
    return poly_disp[k] + u * d[k]

def sample_fracs_by_pixel(poly_disp, n, margin_px, sep_px):
    d = np.diff(poly_disp, axis=0)
    seg = np.hypot(d[:,0], d[:,1]); total = seg.sum()
    if total == 0: return [0.5] * n
    usable = max(0.0, total - 2*margin_px)
    if n == 1:
        targets = [total/2.0]
    else:
        span = (n-1) * sep_px
        if span > usable: sep_px = usable / max(1, n-1)
        start = (total - (n-1)*sep_px) / 2.0
        targets = [start + i*sep_px for i in range(n)]
    return [t/total for t in targets]

# ---------------------- 轻量“排斥式”松弛：把过近节点推开 ----------------------
def relax_positions(pos, min_dist=NODE_MIN_DIST, iters=REPEL_ITERS, step=REPEL_STEP,
                    extra_pairs=EXTRA_REPEL_PAIRS, extra_strength=EXTRA_REPEL_STRENGTH):
    nodes = list(pos.keys())
    P = np.array([pos[n] for n in nodes], dtype=float)
    n = len(nodes)
    for _ in range(iters):
        disp = np.zeros_like(P)
        for i in range(n-1):
            for j in range(i+1, n):
                delta = P[j] - P[i]
                dist = np.linalg.norm(delta) + 1e-9
                if dist < min_dist:
                    force = (min_dist - dist) / min_dist
                    d = delta / dist
                    w = 1.0
                    if ((nodes[i], nodes[j]) in extra_pairs) or ((nodes[j], nodes[i]) in extra_pairs):
                        w *= extra_strength
                    disp[i] -= d * force * w
                    disp[j] += d * force * w
        P += step * disp
    # 约束到 ~单位圆
    rmax = max(1.0, np.max(np.linalg.norm(P, axis=1)))
    P = P / rmax
    for n_, p in zip(nodes, P):
        pos[n_] = p
    return pos

# ---------------------- 指定节点对的“最小角度分开”硬约束 ----------------------
def enforce_angular_spread(pos, pairs=ANGULAR_SPREAD_PAIRS,
                           min_angle_deg=PAIR_MIN_ANGLE_DEG,
                           radius_mode=PAIR_RADIUS_MODE):
    min_angle = np.deg2rad(min_angle_deg)
    for a, b in list(pairs):
        if a not in pos or b not in pos: continue
        xa, ya = pos[a]; xb, yb = pos[b]
        ta, tb = math.atan2(ya, xa), math.atan2(yb, xb)

        # 把角度差规范到 [0, pi]
        diff = abs((tb - ta + math.pi) % (2*math.pi) - math.pi)
        if diff >= min_angle:
            continue  # 已满足

        # 以“平均角度”为中心左右对称展开
        # 用复数平均避免 2π 跳变
        ca, cb = np.exp(1j*ta), np.exp(1j*tb)
        t_avg = np.angle((ca + cb) / 2.0)
        t1 = t_avg - min_angle/2.0
        t2 = t_avg + min_angle/2.0

        ra, rb = np.hypot(xa, ya), np.hypot(xb, yb)
        if radius_mode == "max":
            r = max(ra, rb)
            ra = rb = r

        pos[a] = np.array([ra * math.cos(t1), ra * math.sin(t1)])
        pos[b] = np.array([rb * math.cos(t2), rb * math.sin(t2)])
    return pos

# ========================== 绘图函数 ==========================
def plot_cell_communication(module, module_cell_types, module_ligand_receptors,
                            module_interactions, cell_max_weights):
    active = {d["sender"] for d in module_interactions[module]} | {d["receiver"] for d in module_interactions[module]}
    if not active:
        print(f"跳过模块 {module} - 无有效相互作用")
        return

    fig, ax = plt.subplots(figsize=(14, 12))
    G = nx.DiGraph()

    # 节点
    for cell in active:
        ct2 = celltype_mapping.get(cell, "Unknown")
        color = celltype_2_colors.get(ct2, "#CCCCCC")
        size  = 1500 + cell_max_weights.get(cell, 0) * 5000
        G.add_node(cell, color=color, celltype_2=ct2, size=size)

    # 边
    lr_colors, idx = {}, 0
    for d in module_interactions[module]:
        u, v, lr = d["sender"], d["receiver"], d["lr_pair"]
        if lr not in lr_colors:
            lr_colors[lr] = shoupeitidui[idx % len(shoupeitidui)]; idx += 1
        G.add_edge(u, v, lr_pair=lr, color=lr_colors[lr], groups=",".join(d["groups_present"]))

    # 初始布局 + 排斥松弛 + 对指定节点对做“最小角度分开”
    pos = nx.circular_layout(G, scale=1.0)
    pos = relax_positions(pos)
    pos = enforce_angular_spread(pos)   # ★ 关键一步：强制把 JUN 与 XCL1 分开

    node_colors = [G.nodes[n]['color'] for n in G.nodes()]
    node_sizes  = [G.nodes[n]['size']  for n in G.nodes()]
    nx.draw_networkx_nodes(
        G, pos, node_size=node_sizes, node_color=node_colors,
        alpha=0.9, edgecolors='black', linewidths=NODE_BORDER_WIDTH, ax=ax
    )
    nx.draw_networkx_labels(G, pos, font_size=NODE_LABEL_FONT_SIZE, font_weight='bold', ax=ax)

    # 多条弯边
    edge_counts = defaultdict(int); edge_artists = {}
    for u, v in G.edges(): edge_counts[(u, v)] += 1
    edge_index = defaultdict(int)

    for u, v, data in G.edges(data=True):
        key = (u, v); cnt = edge_counts[key]
        rad = 0.3 * (edge_index[key] - (cnt - 1) / 2.0) if cnt > 1 else 0.05
        edge_index[key] += 1

        groups_abbr = data['groups'].split(',') if data['groups'] else []
        width  = EDGE_WIDTH

        artists = nx.draw_networkx_edges(
            G, pos, edgelist=[(u, v)], edge_color=data['color'], width=width,
            alpha=EDGE_ALPHA, arrows=True, arrowstyle=ARROWSTYLE, arrowsize=ARROWSIZE,
            connectionstyle=f'arc3,rad={rad}',
            min_source_margin=25, min_target_margin=25, node_size=node_sizes, ax=ax
        )
        edge_artists[(u, v)] = artists[0]  # FancyArrowPatch

    # 强制渲染，得到边几何
    fig.canvas.draw()

    # 在边上放置组别小符号（沿边 / 法线堆叠）
    for (u, v), patch in edge_artists.items():
        data = G[u][v]
        groups_abbr = data['groups'].split(',') if data['groups'] else []
        if not groups_abbr: continue

        path_disp = patch.get_path().transformed(patch.get_transform())
        polys_disp = path_disp.to_polygons(closed_only=False)
        if not polys_disp: continue
        main_curve_disp = polys_disp[0]
        main_curve_data = ax.transData.inverted().transform(main_curve_disp)

        L_px = polyline_length_px(main_curve_disp)
        n = len(groups_abbr)
        need_span = (n-1)*DOT_SEP_PX + 2*END_MARGIN_PX
        can_inline = (L_px >= max(MIN_INLINE_PX, need_span))

        if can_inline:
            fracs = sample_fracs_by_pixel(main_curve_disp, n, END_MARGIN_PX, DOT_SEP_PX)
            for frac, abbr in zip(fracs, groups_abbr):
                p = point_on_polyline_by_frac(main_curve_data, frac)
                mk = group_markers.get(abbr, "o")

                if mk in LINE_MARKERS:
                    ax.scatter([p[0]], [p[1]], s=DOT_SIZE*HALO_SCALE, marker=mk,
                               facecolors='none', edgecolors='white',
                               linewidths=HALO_EDGEWIDTH*1.6, zorder=7, clip_on=False)
                    ax.scatter([p[0]], [p[1]], s=DOT_SIZE, marker=mk,
                               facecolors='none', edgecolors=group_colors.get(abbr, '#444444'),
                               linewidths=DOT_EDGEWIDTH+0.4, zorder=8, clip_on=False)
                else:
                    ax.scatter([p[0]], [p[1]], s=DOT_SIZE*HALO_SCALE, marker=mk,
                               c=['white'], edgecolors='white',
                               linewidths=HALO_EDGEWIDTH, zorder=7, clip_on=False)
                    ax.scatter([p[0]], [p[1]], s=DOT_SIZE, marker=mk,
                               c=[group_colors.get(abbr, '#444444')],
                               edgecolors='black', linewidths=DOT_EDGEWIDTH,
                               zorder=8, clip_on=False)
        else:
            # 法线堆叠
            mid_frac = 0.5
            pm = point_on_poly_by_frac_disp(main_curve_disp, mid_frac)
            p2 = point_on_poly_by_frac_disp(main_curve_disp, min(0.999, mid_frac + 0.01))
            tan = p2 - pm; L = np.hypot(tan[0], tan[1])
            npx, npy = (0.0, 1.0) if L == 0 else (-tan[1]/L, tan[0]/L)

            bbox = ax.get_window_extent()
            cx, cy = bbox.x0 + bbox.width/2.0, bbox.y0 + bbox.height/2.0
            to_center = np.array([cx - pm[0], cy - pm[1]])
            if npx*to_center[0] + npy*to_center[1] > 0: npx, npy = -npx, -npy

            pm_data = ax.transData.inverted().transform(pm)
            start_offset = - (n-1)/2.0 * STACK_SEP_PX
            for i, abbr in enumerate(groups_abbr):
                off_px = start_offset + i*STACK_SEP_PX
                dv = px_to_data_vec(ax, pm, off_px*npx, off_px*npy)
                p = pm_data + dv
                mk = group_markers.get(abbr, "o")

                if mk in LINE_MARKERS:
                    ax.scatter([p[0]], [p[1]], s=DOT_SIZE*HALO_SCALE, marker=mk,
                               facecolors='none', edgecolors='white',
                               linewidths=HALO_EDGEWIDTH*1.6, zorder=7, clip_on=False)
                    ax.scatter([p[0]], [p[1]], s=DOT_SIZE, marker=mk,
                               facecolors='none', edgecolors=group_colors.get(abbr, '#444444'),
                               linewidths=DOT_EDGEWIDTH+0.4, zorder=8, clip_on=False)
                else:
                    ax.scatter([p[0]], [p[1]], s=DOT_SIZE*HALO_SCALE, marker=mk,
                               c=['white'], edgecolors='white',
                               linewidths=HALO_EDGEWIDTH, zorder=7, clip_on=False)
                    ax.scatter([p[0]], [p[1]], s=DOT_SIZE, marker=mk,
                               c=[group_colors.get(abbr, '#444444')],
                               edgecolors='black', linewidths=DOT_EDGEWIDTH,
                               zorder=8, clip_on=False)

    # 图例：受配体对
    lr_legend = []
    lr_colors_used = {}
    for _, _, ed in G.edges(data=True):
        lr_colors_used[ed['lr_pair']] = ed['color']
    for lr, color in lr_colors_used.items():
        lr_legend.append(mlines.Line2D([], [], color=color, marker='_',
                                       markersize=LEGEND_MARKERSIZE+6,
                                       label=lr, linewidth=LEGEND_LR_LINEWIDTH))

    # 图例：细胞大类
    ct2_legend = [mlines.Line2D([], [], color=celltype_2_colors.get(ct2, "#CCCCCC"),
                                marker='o', markersize=LEGEND_MARKERSIZE,
                                label=ct2, linestyle='None')
                  for ct2 in {G.nodes[n]['celltype_2'] for n in G.nodes()}]

    ax.legend(handles=(lr_legend + ct2_legend),
              loc='upper left', bbox_to_anchor=(1, 1), fontsize=LEGEND_FONT_SIZE,
              frameon=False)
    ax.set_title(f'Cell Communication Network for Module: {module}', fontsize=TITLE_FONT_SIZE)
    ax.axis('off')

    os.makedirs(save_dir, exist_ok=True)
    plt.savefig(os.path.join(save_dir, f"module_{module}_network.pdf"),
                bbox_inches='tight', dpi=600)
    plt.show()

# ========================== 批量绘图 ==========================
start_time = time.time()
print("开始绘制模块网络图...")
plot_group_symbol_legend(save_dir)  # 疾病组符号图注所有模块一致，单独保存一次
for module in modules:
    if module in module_interactions and module_interactions[module]:
        print(f"绘制模块: {module} (包含 {len(module_interactions[module])} 个相互作用)")
        plot_cell_communication(module, module_cell_types, module_ligand_receptors,
                                module_interactions, module_cell_max_weights[module])
    else:
        print(f"跳过模块 {module} - 未找到有效的相互作用")
print(f"所有模块网络图绘制完成，总耗时: {time.time()-start_time:.2f}秒")